<a href="https://colab.research.google.com/github/seonilj/tribev2-exploration/blob/main/notebooks/02_vertex_atlas_mapping.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cortical Surface Vertex-to-Atlas Anatomical Mapping via Meta AI's TRIBE v2
*대뇌 피질 표면 정점-아틀라스 해부학적 매핑*

## 📌 Project Overview
This notebook implements an automated neuroanatomical localization pipeline that maps raw cortical surface vertex indices predicted by Meta AI's `tribev2` model onto concrete, interpretable brain regions.

The core objective is to translate abstract mathematical spatial tokens (`fsaverage5` vertices) into anatomical labels (e.g., *precuneus, superiortemporal*) by aligning the neural prediction array with standard cortical parcellation maps. This step bridges the gap between raw machine learning signal predictions and biological neuroscience interpretation.

본 노트북은 Meta AI의 `tribev2` 모델이 예측한 피질 표면 버텍스(Vertex) 인덱스를 실제 뇌 해부학적 영역 명칭으로 자동 매핑하는 파이프라인을 구현합니다. `fsaverage5` 표준 메쉬 공간의 점들을 Desikan-Killiany/Destrieux 아틀라스와 정렬함으로써, 모델 예측치로부터 생물학적 해석을 도출하는 핵심 다리 역할을 합니다.

## 🛠️ Key Technical Features


## 1. Environment Preparation, Dependency, and Libraries
Installing crucial neuroimaging toolkits (`nibabel`, `nilearn`, `mne`) to resolve spatial mapping dependencies and importing primary mathematical frameworks (`numpy`, `pandas`).
* 원활한 공간 매핑과 데이터 분석을 위해 관련 필수 오픈소스 패키지를 설치하고 기본 핵심 라이브러리를 임포트합니다.

In [ ]:
# Clean up pre-installed PyTorch packages to avoid dependency conflicts
!pip uninstall -y torch torchaudio torchvision torchtext

Found existing installation: torch 2.11.0+cu128
Uninstalling torch-2.11.0+cu128:
  Successfully uninstalled torch-2.11.0+cu128
Found existing installation: torchaudio 2.11.0+cu128
Uninstalling torchaudio-2.11.0+cu128:
  Successfully uninstalled torchaudio-2.11.0+cu128
Found existing installation: torchvision 0.26.0+cu128
Uninstalling torchvision-0.26.0+cu128:
  Successfully uninstalled torchvision-0.26.0+cu128


In [ ]:
# fix: conflict of PyTorch(torch)와 Torchaudio package versions
# solution: 구글 Colab 환경에서 torch와 torchaudio를 호환되는 최신 버전으로 동시에 업데이트
# !pip install --upgrade torch torchaudio

# Install official PyTorch binaries configured for CUDA 12.4
!pip install torch torchaudio torchvision --index-url https://pytorch.org

Looking in indexes: https://pytorch.org
ERROR: Could not find a version that satisfies the requirement torch (from versions: none)
ERROR: No matching distribution found for torch


In [ ]:
# Pin NumPy version (<2.0.0) and install Meta's Tribe v2 repository with plotting tools
!pip install -q -U "numpy<2.0.0"
!uv pip install -q "tribev2[plotting] @ git+https://github.com/facebookresearch/tribev2.git"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 3.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 56.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
accelerate 1.14.0 requires torch>=2.0.0, which is not installed.
fastai 2.8.7 requires torch<3,>=1.10, which is not installed.
fastai 2.8.7 requires torchvision>=0.11, which is not installed.
peft 0.19.1 requires torch>=1.13.0, which is not installed.
sentence-transformers 5.6.0 requires torch>=1.11.0, which is not installed.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which 

In [ ]:
# neuroimaging packages in Colab environment
!pip install nibabel nilearn mne

In [ ]:
# import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import nibabel as nib
from nilearn import datasets, surface
from IPython.display import display
from pathlib import Path

## 2. Load Core Structural Templates (`fsaverage5` Surface Mesh)
Fetching standard `fsaverage5` pial and inflated mesh coordinate representations to establish a baseline spatial topology for index positioning.
* 점(vertex) 위치의 기초 공간 토폴로지를 구축하기 위해 Nilearn을 사용하여 fsaverage5 표준 뇌 표면 데이터를 다운로드합니다.

In [ ]:
fsaverage = datasets.fetch_surf_fsaverage(mesh='fsaverage5')

# display the layout structure of fsaverage5
for key, value in fsaverage.items():
    print(f"{key}: {str(value)[:60]}...")

area_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
area_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
curv_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
curv_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
flat_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
flat_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
infl_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
infl_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
pial_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
pial_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
sphere_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
sphere_right: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
sulc_left: /usr/local/lib/python3.12/dist-packages/nilearn/datasets/dat...
sulc_right: /us

## 3. Extract Surface Geometry and Cortical Atlas Alignment
Using `nilearn.surface.load_surf_mesh` to securely parse GIFTI coordinates without file-format parsing exceptions, and loading the Destrieux atlas structures.
* 구글 코랩 환경에서의 GIFTI 파일 구조적 에러를 완벽하게 차단하는 전용 표면 메쉬 로더를 연결하고, 뇌 부위별 이름이 들어있는 아틀라스 레이블을 정렬합니다.

In [ ]:
# left hemisphere
lh_mesh = surface.load_surf_mesh(fsaverage['infl_left'])
lh_coords, lh_faces = lh_mesh.coordinates, lh_mesh.faces
print(f"Left Hemisphere Inflated Coordinates Shape: {lh_coords.shape}")

# right hemisphere
rh_mesh = surface.load_surf_mesh(fsaverage['infl_right'])
rh_coords, rh_faces = rh_mesh.coordinates, rh_mesh.faces
print(f"Right Hemisphere Inflated Coordinates Shape: {rh_coords.shape}")

Left Hemisphere Inflated Coordinates Shape: (10242, 3)
Right Hemisphere Inflated Coordinates Shape: (10242, 3)


In [ ]:
# check FreeSurfer label directory
print("---Fetching Cortical Surface Atlas ---")
atlas_surf = datasets.fetch_atlas_surf_destrieux()

# extract the labels (brain region names)
region_names = atlas_surf['labels']
print(f"Total atlas regions loaded: {len(region_names)}")

---Fetching Cortical Surface Atlas ---


[fetch_atlas_surf_destrieux] Added README.md to /root/nilearn_data

[fetch_atlas_surf_destrieux] Dataset created in /root/nilearn_data/destrieux_surface

[fetch_atlas_surf_destrieux] Downloading data from 
https://www.nitrc.org/frs/download.php/9343/lh.aparc.a2009s.annot ...

[fetch_atlas_surf_destrieux]  ...done. (1 seconds, 0 min)

[fetch_atlas_surf_destrieux] Downloading data from 
https://www.nitrc.org/frs/download.php/9342/rh.aparc.a2009s.annot ...

[fetch_atlas_surf_destrieux]  ...done. (0 seconds, 0 min)

Total atlas regions loaded: 76


/tmp/ipykernel_1887/3044915610.py:3: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index    name
     0 Unknown

  atlas_surf = datasets.fetch_atlas_surf_destrieux()
/tmp/ipykernel_1887/3044915610.py:3: UserWarning: 
The following regions are present in the atlas look-up table,
but missing from the atlas image:

 index    name
     0 Unknown

  atlas_surf = datasets.fetch_atlas_surf_destrieux()


In [ ]:
# Desikan-Killiany Atlas
# Nilearn provides pre-loaded vertex arrays matching fsaverage5
lh_labels = atlas_surf['map_left']
rh_labels = atlas_surf['map_right']

# destrieux labels use standard string decoding natively from Nilearn
# storing clean references for the mapping function
lh_names = region_names
rh_names = region_names

print(f"LH Label Array Shape: {lh_labels.shape}")
print(f"RH Label Array Shape: {rh_labels.shape}")

LH Label Array Shape: (10242,)
RH Label Array Shape: (10242,)
